# Data preparation → master file for IS normalization

What changed so the output works with `a_0.0._DataAnalysis_Normalization_preparation.ipynb`:
- **13C internal standards are kept** as raw areas (no imputation). An IS missing in > `IS_MAX_MISSING` of injections (not detected in that mode) is dropped.
- **Metabolite names are cleaned of stray characters** (`ASPARAGINEÂ·` → `ASPARAGINE`).
- **Column prefixes are `Pos_` / `Neg_`** (set in `MODE_PREFIX`), because that's what the normalization script looks for.
- **`Type` is set from the sample name**: `'Quality control'` for QCs and `'Sample'` for everything else, so no row is blank.
- **New file `master_analysis_with_QC2.csv`** (not `master_analysis_file2.csv`, which is samples only) holds samples + the 19 QCs in one table, sorted by injection #. This is the input for IS normalization.
- **`Pr_concentration`** is a numeric copy of `Pr_Con` (`'1.8 mg/ml'` → 1.8).
- **Batch segment is rebuilt from injection #** (`BATCH_B_FROM_INJECTION`), which fills the 2 blank rows and gives the QCs a batch.
- **`DROP_BY_MODE` moved into the parameter block.** Before, `load_matrix` used it before it was defined, so a fresh kernel crashed with a NameError.


In [ ]:
# ==============================================================================
# Metabolomics data preparation (25-M113) - ONE CELL, all parameters on top
# Output is ready for the IS-normalization notebook
#   (a_0.0._DataAnalysis_Normalization_preparation.ipynb)
# ==============================================================================
import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# 1. PARAMETERS - change only this block
# ------------------------------------------------------------------------------
META_FILE = r"O:\metabolom\Result\Core_results_original_data\25-M-113\metaData_NN.xlsx"
POS_FILE  = r"O:\metabolom\Result\Core_results_original_data\25-M-113\raw data\new version_09.2026\25-M113-pos-export matrix-area2.xlsx"
NEG_FILE  = r"O:\metabolom\Result\Core_results_original_data\25-M-113\raw data\new version_09.2026\25-M113-neg-matrics-area-revised2.xlsx"
OUT_DIR   = r"O:\metabolom\Result\new version result_09.2026"

META_ID_COL    = 'Sample ID'     # sample column in metadata
METAB_NAME_COL = 'Sample Name'   # metabolite-name column in pos/neg files
INJECTION_COL  = 'injection #'   # 'injection_12' -> 12

QC_PATTERN      = r'QC'                 # pooled QC samples
EXCLUDE_PATTERN = r'HB|Blank|blank'     # standards mixes and blanks -> removed

# sample names that differ between files (after '-' -> '_'); right side = correct name
RENAME_SAMPLES = {
    '55H_AaspartateP4_M_KI_52': '55H_AP4_M_KI_52',
    '86H_AP4_M_KI_9':           '86H_AP4_M_KI_98',
    'QC_25':                    'QC 25_M113',
}
DROP_SAMPLES = []             # e.g. ['55H_AP4_M_KI_52'] (labelled 'Standard' in pos file)

# metabolites removed per mode (see markdown note below)  -- must be defined BEFORE load_matrix runs
DROP_BY_MODE = {'NEG': ['SUCROSE', 'ACONITATE'], 'POS': []}

# ---- settings the IS-normalization notebook depends on ----
MODE_PREFIX  = {'POS': 'Pos_', 'NEG': 'Neg_'}   # column prefix; normalization looks for 'Pos_' / 'Neg_'
IS_PATTERN   = r'^13C '        # spiked 13C internal standards: KEPT, not imputed
IS_MAX_MISSING = 0.50         # drop an IS if missing in > 50 % of injections (not detected in this mode)
TYPE_COL     = 'Type'
TYPE_SAMPLE  = 'Sample'
TYPE_QC      = 'Quality control'   # normalization selects df['Type'] == 'Quality control'
PR_COL       = 'Pr_Con'            # text like '1.8 mg/ml'
PR_NUM_COL   = 'Pr_concentration'  # numeric copy used for protein normalization
BATCH_COL    = 'Batch segment( before jump=A, after jump=B)'
BATCH_B_FROM_INJECTION = 120       # injection # >= this -> 'B', else 'A' (jump is between 119 and 125); None = keep metadata values

MAX_MISSING = 0.80         # drop metabolite if missing in > 80 % of biological samples
QC_CV_MAX   = None         # e.g. 0.30 -> drop metabolites with CV > 30 % in QCs; None = off
IMPUTE      = 'half_min'   # 'half_min', 'min', or None

RENAME_META_COLS = {'Pr_Con.': 'Pr_Con'}
GROUP_COLS       = {'Group_Region': 'Region of brain', 'Group_APOE': 'line of APOE'}

# ------------------------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------------------------
def std_name(s):
    """'-' -> '_', strip spaces, QC1 -> QC_1, then apply RENAME_SAMPLES."""
    s = str(s).strip().replace('-', '_')
    s = pd.Series([s]).str.replace(r'^QC\s*_?(\d+)$', r'QC_\1', regex=True)[0]
    return RENAME_SAMPLES.get(s, s)

def load_matrix(path, mode):
    """Excel export -> numeric table (rows = metabolites incl. 13C IS, columns = samples)."""
    df = pd.read_excel(path)
    df.columns = [c if c == METAB_NAME_COL else std_name(c) for c in df.columns]
    types = df.loc[df[METAB_NAME_COL] == 'Sample Type'].iloc[:1, 1:].squeeze(axis=0)
    odd = types[types != 'Unknown'] if len(types) else types
    if len(odd):
        print(f"[{mode}] Sample Type not 'Unknown': {odd.to_dict()}")
    df = df[df[METAB_NAME_COL] != 'Sample Type']
    df = df.dropna(subset=[METAB_NAME_COL]).set_index(METAB_NAME_COL)
    df.index = (df.index.str.replace(r'[^\x20-\x7E]', '', regex=True)   # 'ASPARAGINEÂ·' -> 'ASPARAGINE'
                        .str.strip())
    df = df.drop(index=[m for m in DROP_BY_MODE.get(mode, []) if m in df.index])
    df = df.apply(pd.to_numeric, errors='coerce').replace(0, np.nan)
    df = df.drop(columns=df.columns[df.columns.str.contains(EXCLUDE_PATTERN)])
    return df.drop(columns=[c for c in DROP_SAMPLES if c in df.columns])

def filter_and_impute(df, mode):
    """Split off 13C IS (kept raw), filter + impute the real metabolites.
    Returns (bio, qc) with rows = features incl. IS, index prefixed with MODE_PREFIX."""
    is_mask = df.index.str.contains(IS_PATTERN)
    is_df, df = df[is_mask], df[~is_mask]

    qc_cols  = df.columns[df.columns.str.contains(QC_PATTERN)]
    bio_cols = df.columns.difference(qc_cols, sort=False)
    n0 = len(df)
    df = df[df[bio_cols].isna().mean(axis=1) <= MAX_MISSING]          # missingness in samples only
    n1 = len(df)
    if QC_CV_MAX is not None and len(qc_cols):                        # QC reproducibility
        df = df[df[qc_cols].std(axis=1) / df[qc_cols].mean(axis=1) <= QC_CV_MAX]
    still = df[bio_cols].isna().sum(axis=1)
    still = still[still > 0]
    if IMPUTE in ('half_min', 'min'):
        fill = df[bio_cols].min(axis=1) * (0.5 if IMPUTE == 'half_min' else 1.0)
        df = df.apply(lambda row: row.fillna(fill[row.name]), axis=1)
    print(f"[{mode}] metabolites: {n0} -> {n1} (<= {MAX_MISSING:.0%} missing) -> {len(df)} (QC-CV)"
          f" | imputed: {still.to_dict()}")

    is_miss = is_df.isna().mean(axis=1)
    dropped = list(is_miss[is_miss > IS_MAX_MISSING].index)
    is_df   = is_df[is_miss <= IS_MAX_MISSING]
    is_nan  = is_df.isna().sum(axis=1)
    print(f"[{mode}] internal standards kept ({len(is_df)}): {list(is_df.index)}"
          f" | missing values: {is_nan[is_nan > 0].to_dict() or 'none'}")
    print(f"[{mode}] internal standards dropped (not detected, > {IS_MAX_MISSING:.0%} missing): {dropped or 'none'}")

    df = pd.concat([df, is_df])
    df.index = MODE_PREFIX[mode] + df.index
    return df[bio_cols], df[qc_cols]

# ------------------------------------------------------------------------------
# 3. LOAD + CLEAN
# ------------------------------------------------------------------------------
meta_all = pd.read_excel(META_FILE).rename(columns=RENAME_META_COLS).dropna(subset=[META_ID_COL])
meta_all[META_ID_COL] = meta_all[META_ID_COL].map(std_name)
if INJECTION_COL in meta_all.columns:
    meta_all[INJECTION_COL] = pd.to_numeric(meta_all[INJECTION_COL].astype(str)
                                            .str.extract(r'(\d+)', expand=False), errors='coerce')
meta_all = meta_all[~meta_all[META_ID_COL].str.contains(EXCLUDE_PATTERN)
                    & ~meta_all[META_ID_COL].isin(DROP_SAMPLES)].copy()
is_qc = meta_all[META_ID_COL].str.contains(QC_PATTERN)

# Type: set from the sample name, so QC rows are always 'Quality control' and no row is blank
meta_all[TYPE_COL] = np.where(is_qc, TYPE_QC, TYPE_SAMPLE)

# Protein concentration: '1.8 mg/ml' -> 1.8 (numeric copy; original text kept)
if PR_COL in meta_all.columns:
    meta_all[PR_NUM_COL] = pd.to_numeric(meta_all[PR_COL].astype(str)
                                         .str.extract(r'([\d.]+)', expand=False), errors='coerce')

# Batch segment from injection order (fills the blanks, also gives QCs a batch)
if BATCH_B_FROM_INJECTION is not None and INJECTION_COL in meta_all.columns:
    meta_all[BATCH_COL] = np.where(meta_all[INJECTION_COL] >= BATCH_B_FROM_INJECTION, 'B', 'A')
    meta_all.loc[meta_all[INJECTION_COL].isna(), BATCH_COL] = np.nan

meta    = meta_all[~is_qc]          # biological samples
meta_qc = meta_all[is_qc]           # QC samples (injection #, batch, ...)

pos_bio, pos_qc = filter_and_impute(load_matrix(POS_FILE, 'POS'), 'POS')
neg_bio, neg_qc = filter_and_impute(load_matrix(NEG_FILE, 'NEG'), 'NEG')

# ------------------------------------------------------------------------------
# 4. COMBINE POS + NEG AND MERGE WITH METADATA
# ------------------------------------------------------------------------------
features    = pd.concat([pos_bio, neg_bio]).T.rename_axis(META_ID_COL)   # rows = samples
qc_features = pd.concat([pos_qc,  neg_qc]).T.rename_axis(META_ID_COL)
all_cols     = list(features.columns)
is_cols      = [c for c in all_cols if '13C' in c]
feature_cols = [c for c in all_cols if '13C' not in c]      # real metabolites only

for label, m, f in [('samples', meta, features), ('QC', meta_qc, qc_features)]:
    print(f"\n[{label}] only in metadata    : {sorted(set(m[META_ID_COL]) - set(f.index)) or 'none'}")
    print(f"[{label}] only in metabolomics: {sorted(set(f.index) - set(m[META_ID_COL])) or 'none'}")

master = meta.merge(features.reset_index(), on=META_ID_COL, how='inner')
for new, old in GROUP_COLS.items():
    master[new] = master[old].astype(str).str.strip()

qc_master = meta_qc.merge(qc_features.reset_index(), on=META_ID_COL, how='right')
qc_master[TYPE_COL] = TYPE_QC                                # also for QCs missing from metadata

# samples + QC in one table -> input for the IS-normalization notebook
master_with_qc = pd.concat([master, qc_master], ignore_index=True)
if INJECTION_COL in master_with_qc.columns:
    master_with_qc = master_with_qc.sort_values(INJECTION_COL, ignore_index=True)

# ------------------------------------------------------------------------------
# 5. CHECKS + SAVE
# ------------------------------------------------------------------------------
n_pos = sum(c.startswith(MODE_PREFIX['POS']) for c in feature_cols)
n_neg = sum(c.startswith(MODE_PREFIX['NEG']) for c in feature_cols)
print(f"\nMaster: {len(master)} samples x {len(feature_cols)} metabolites ({n_pos} POS + {n_neg} NEG)"
      f" + {len(is_cols)} IS | QC: {len(qc_master)} QC samples")
print("IS columns:", is_cols)
print("NaN left in metabolites:", int(master_with_qc[feature_cols].isna().sum().sum()))
print("NaN in IS columns      :", int(master_with_qc[is_cols].isna().sum().sum()))
print(f"{TYPE_COL}:", master_with_qc[TYPE_COL].value_counts(dropna=False).to_dict())
if BATCH_COL in master_with_qc.columns:
    print("Batch:", master_with_qc.groupby(TYPE_COL)[BATCH_COL].value_counts(dropna=False).to_dict())
if PR_NUM_COL in master.columns:
    bad_pr = master.loc[~(master[PR_NUM_COL] > 0), META_ID_COL].tolist()
    print(f"{PR_NUM_COL}: min={master[PR_NUM_COL].min()}, max={master[PR_NUM_COL].max()}"
          f" | missing/non-positive in samples: {bad_pr or 'none'}")
if INJECTION_COL in master_with_qc.columns:
    dup = master_with_qc[master_with_qc[INJECTION_COL].duplicated(keep=False)]
    if len(dup):
        print("WARNING duplicated injection #:", dup.set_index(META_ID_COL)[INJECTION_COL].to_dict())
for new in GROUP_COLS:
    print(f"{new}: {master[new].value_counts().to_dict()}")

assert (master_with_qc[TYPE_COL] == TYPE_QC).sum() > 0, "No QC rows - normalization needs them"
assert len(is_cols) > 0, "No 13C internal standards - check IS_PATTERN"

master.to_csv(os.path.join(OUT_DIR, 'master_analysis_file2.csv'), index=False)                 # samples only
qc_master.to_csv(os.path.join(OUT_DIR, 'qc_features2.csv'), index=False)                        # QC only
master_with_qc.to_csv(os.path.join(OUT_DIR, 'master_analysis_with_QC2.csv'), index=False)       # -> normalization
print(f"\nSaved to {OUT_DIR}:")
print(f"  master_analysis_file2.csv    : {len(master)} samples only      -> statistics")
print(f"  qc_features2.csv             : {len(qc_master)} QC only")
print(f"  master_analysis_with_QC2.csv : {len(master_with_qc)} samples + QC  -> USE THIS for IS normalization")


### the metabolites had been dropped:
- Sorbitol: it was recognized as Trehalose on the Library with high purity -> it made deciseion to remove this metabolite
- Aconita : purity of library was very low and the peak shape was weird -> decided to remove it from the metabolites list

(`DROP_BY_MODE` is now set in the parameter block of the cell above.)


In [ ]:
# ------------------------------------------------------------------------------
# Which samples had NaN (before imputation) in the kept metabolites?
# ------------------------------------------------------------------------------
raw = pd.concat([load_matrix(POS_FILE, 'POS').rename(index=lambda m: MODE_PREFIX['POS'] + m),
                 load_matrix(NEG_FILE, 'NEG').rename(index=lambda m: MODE_PREFIX['NEG'] + m)])
raw = raw.loc[feature_cols, features.index]              # kept metabolites, biological samples only
nan_mask = raw.isna().rename_axis(index='Metabolite', columns=META_ID_COL)

# 1) long table: one row per missing cell, with the sample's groups
nan_list = (nan_mask.stack().loc[lambda s: s].reset_index()[['Metabolite', META_ID_COL]]
            .merge(master[[META_ID_COL] + list(GROUP_COLS)], on=META_ID_COL, how='left'))

# 2) per metabolite: how many samples, and in which groups
for met, grp in nan_list.groupby('Metabolite', sort=False):
    print(f"\n{met}: NaN in {len(grp)} of {raw.shape[1]} samples")
    for g in GROUP_COLS:
        print(f"   {g}: {grp[g].value_counts().to_dict()}")
    print("   samples:", ', '.join(grp[META_ID_COL]))

nan_list.to_csv(os.path.join(OUT_DIR, 'nan_samples_before_imputation.csv'), index=False)
print(f"\nSaved nan_samples_before_imputation.csv ({len(nan_list)} missing cells) to {OUT_DIR}")
